In [117]:
import pandas as pd
import numpy as np

In [118]:
balls = pd.read_parquet("../ml-service/data/processed/v3_beta/clean_deliveries.parquet")
matches = pd.read_parquet("../ml-service/data/processed/v3_beta/clean_matches.parquet")

In [119]:
balls.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,0,0,0,0.0,1.0,0.0,Not Out,2008-04-18,1.0
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,1,0,0.0,0.0,0.0,Not Out,2008-04-18,1.0
3,335982,0,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0
4,335982,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0


In [120]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs'],
      dtype='object')

In [121]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277161
2        32
Name: count, dtype: int64

In [122]:
duplicates = balls[
    balls.duplicated(['matchId','inning','over','ball'], keep=False)
].sort_values(['matchId','inning','over','ball'])
duplicates.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs
5635,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.0,0.0,0.0,Not Out,2008-05-04,0.0
5636,336005,1,6,1,Rajasthan Royals,Chennai Super Kings,GC Smith,SA Asnodkar,JA Morkel,0,0,0,0.0,4.0,0.0,Not Out,2008-05-04,4.0
9339,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,0,1,0,0.0,0.0,0.0,Not Out,2008-05-18,1.0
9340,336024,0,18,1,Mumbai Indians,Deccan Chargers,PR Shah,YV Takawale,DP Vijaykumar,1,0,0,0.0,0.0,0.0,Not Out,2008-05-18,1.0
35713,419141,1,1,1,Deccan Chargers,Rajasthan Royals,AC Gilchrist,VVS Laxman,SR Watson,0,1,0,0.0,0.0,0.0,Not Out,2010-04-05,1.0


In [123]:
len(duplicates)

64

In [124]:
balls = (
    balls.groupby(['matchId','inning','over','ball'], as_index=False)
    .agg({
        'batsman_runs': 'sum',
        'isWide': 'sum',
        'isNoBall': 'sum',
        'Byes': 'sum',
        'LegByes': 'sum',
        'Penalty': 'sum',
        'total_runs': 'sum',  # important

        'batting_team': 'first',
        'bowling_team': 'first',
        'batsman': 'first',
        'non_striker': 'first',
        'bowler': 'first',
        'player_dismissed': lambda x: x[x != "Not Out"].iloc[0] if any(x != "Not Out") else "Not Out",
        'date': 'first'
    })
)

In [125]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277193
Name: count, dtype: int64

In [126]:
balls = balls.sort_values(['matchId', 'inning', 'over', 'ball']).reset_index(drop=True)

In [127]:
balls.groupby(['matchId','inning','over','ball']).size().value_counts()

1    277193
Name: count, dtype: int64

In [128]:
mask = balls['isNoBall'] > 1

balls.loc[mask, 'batsman_runs'] += (balls.loc[mask, 'isNoBall'] - 1)
balls.loc[mask, 'isNoBall'] = 1

In [129]:
balls['repeat'] = np.where(balls['isWide'] > 0, balls['isWide'], 1)

balls = balls.loc[balls.index.repeat(balls['repeat'])].copy()

balls.loc[balls['isWide'] > 0, 'isWide'] = 1

balls.drop(columns=['repeat'], inplace=True)

In [130]:
balls['is_legal'] = ((balls['isWide'] == 0) & (balls['isNoBall'] == 0)).astype(int)

balls['ball'] = (
    balls.groupby(['matchId','inning','over'])['is_legal']
    .cumsum()
)

balls['ball'] = balls['ball'].replace(0, np.nan)
balls['ball'] = (
    balls.groupby(['matchId','inning','over'])['ball']
    .ffill()
    .fillna(1)
)

In [131]:
balls = balls.reset_index(drop=True)

In [132]:
balls['isWide'].unique()

array([0, 1])

In [133]:
balls['ball'].unique()

array([1., 2., 3., 4., 5., 6., 7.])

In [134]:
balls = balls[balls['ball'] <= 6]

In [135]:
balls['ball'].unique()

array([1., 2., 3., 4., 5., 6.])

In [136]:
balls.groupby(['matchId','inning','over'])['is_legal'].sum().max()

np.int64(6)

In [137]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal'],
      dtype='object')

In [138]:
balls['legal_ball'] = (balls['isWide'] == 0) & (balls['isNoBall'] == 0)

balls['balls_bowled'] = balls.groupby(['matchId', 'inning'])['legal_ball'].cumsum()

balls['balls_remaining'] = 120 - balls['balls_bowled']

balls.drop(columns=['balls_bowled'], inplace=True)

In [139]:
balls.groupby(['matchId','inning'])['legal_ball'].sum().max()

np.int64(120)

In [140]:
balls = balls.reset_index(drop=True)

In [141]:
balls['balls_remaining'].unique()

array([119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107,
       106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,  94,
        93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,  81,
        80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,  68,
        67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,  55,
        54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,  42,
        41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,  29,
        28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,  16,
        15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,   3,
         2,   1,   0, 120])

In [142]:
balls['legal_ball'][balls['balls_remaining']==120].unique()

array([False])

In [143]:
balls['over_number'] = balls['over'].astype(int) + 1
balls['phase'] = np.select(
    [
        balls['over_number'] <= 6,
        balls['over_number'] <= 15
    ],
    [0, 1],
    default=2
)
# balls.drop(columns=['over_number'], inplace=True)

In [144]:
balls['over_number'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20])

In [145]:
balls[balls['phase']==1].head(5)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,batsman,non_striker,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase
42,335982,0,6,1.0,1,0,0,0.0,0.0,0.0,...,BB McCullum,RT Ponting,AA Noffke,Not Out,2008-04-18,1,True,83,7,1
43,335982,0,6,2.0,1,0,0,0.0,0.0,0.0,...,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,82,7,1
44,335982,0,6,3.0,1,0,0,0.0,0.0,0.0,...,BB McCullum,RT Ponting,AA Noffke,Not Out,2008-04-18,1,True,81,7,1
45,335982,0,6,4.0,2,0,0,0.0,0.0,0.0,...,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,80,7,1
46,335982,0,6,5.0,1,0,0,0.0,0.0,0.0,...,RT Ponting,BB McCullum,AA Noffke,Not Out,2008-04-18,1,True,79,7,1


In [146]:
balls["total_runs"] = (balls["batsman_runs"]+ balls["isWide"]+ balls["isNoBall"]+ balls["Byes"]+ balls["LegByes"]+ balls["Penalty"])
balls['current_score'] = balls.groupby(['matchId', 'inning'])['total_runs'].cumsum()
# balls.drop(columns=['total_runs'], inplace=True)

In [147]:
balls[['over', 'ball','current_score','isWide','isNoBall']].head(10)

,over,ball,current_score,isWide,isNoBall
0,0,1.0,1.0,0,0
1,0,2.0,1.0,0,0
2,0,2.0,2.0,1,0
3,0,3.0,2.0,0,0
4,0,4.0,2.0,0,0
5,0,5.0,2.0,0,0
6,0,6.0,3.0,0,0
7,1,1.0,3.0,0,0
8,1,2.0,7.0,0,0
9,1,3.0,11.0,0,0


In [148]:
balls['current_score'].unique().max()

np.float64(287.0)

In [149]:
balls.loc[balls['Penalty'] == 5, 'batsman_runs'] = 5

In [150]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score'],
      dtype='object')

In [151]:
balls["batsman_runs"] = balls["batsman_runs"] + balls["Byes"] + balls["LegByes"]
balls['batsman_runs'].unique()

array([1., 0., 4., 6., 2., 5., 3., 7.])

In [152]:
balls.loc[balls['batsman_runs'] > 6, 'batsman_runs'] = 6


In [153]:
balls = balls.reset_index(drop=True)

In [154]:
balls['player_dismissed'].head(2)

0    Not Out
1    Not Out
Name: player_dismissed, dtype: object

In [155]:
len(balls['player_dismissed'].unique())

657

In [156]:
balls['is_wicket'] = (balls['player_dismissed'] != 'Not Out').astype(int)
balls['wickets_fallen'] = balls.groupby(['matchId', 'inning'])['is_wicket'].cumsum()
balls.drop(columns=['is_wicket'], inplace=True)

In [157]:
balls['wickets_fallen'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [158]:
balls.head()

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,bowler,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen
0,335982,0,0,1.0,1.0,0,0,0.0,1.0,0.0,...,P Kumar,Not Out,2008-04-18,1,True,119,1,0,1.0,0
1,335982,0,0,2.0,0.0,0,0,0.0,0.0,0.0,...,P Kumar,Not Out,2008-04-18,1,True,118,1,0,1.0,0
2,335982,0,0,2.0,0.0,1,0,0.0,0.0,0.0,...,P Kumar,Not Out,2008-04-18,0,False,118,1,0,2.0,0
3,335982,0,0,3.0,0.0,0,0,0.0,0.0,0.0,...,P Kumar,Not Out,2008-04-18,1,True,117,1,0,2.0,0
4,335982,0,0,4.0,0.0,0,0,0.0,0.0,0.0,...,P Kumar,Not Out,2008-04-18,1,True,116,1,0,2.0,0


In [159]:
balls = balls.reset_index(drop=True)

In [160]:
first_innings_score = (
    balls[balls['inning'] == 0]
    .groupby('matchId')['current_score']
    .max()
)
balls['target'] = balls['matchId'].map(first_innings_score)
balls.loc[balls['inning'] == 1, 'target'] = balls['target'] + 1
balls.loc[balls['inning'] == 0, 'target'] = 0

In [161]:
balls[balls['target']>0].head(2)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,player_dismissed,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target
129,335982,1,0,1.0,1.0,0,0,0.0,0.0,0.0,...,Not Out,2008-04-18,1,True,119,1,0,1.0,0,223.0
130,335982,1,0,1.0,0.0,1,0,0.0,0.0,0.0,...,Not Out,2008-04-18,0,False,119,1,0,2.0,0,223.0


In [162]:
balls['target'].unique().max()

np.float64(288.0)

In [163]:
over_runs = (
    balls.groupby(['matchId', 'inning', 'over_number'])['total_runs']
    .sum()
    .reset_index(name='over_runs')
)
over_runs['last_over_runs'] = (
    over_runs.groupby(['matchId', 'inning'])['over_runs']
    .shift(1)
)
balls = balls.merge(
    over_runs[['matchId', 'inning', 'over_number', 'last_over_runs']],
    on=['matchId', 'inning', 'over_number'],
    how='left'
)
balls['last_over_runs'] = balls['last_over_runs'].fillna(0)
balls['last_over_runs'] = balls['last_over_runs'].astype(int)

In [164]:
balls[['over_number', 'total_runs', 'last_over_runs']].head(10)

,over_number,total_runs,last_over_runs
0,1,1.0,0
1,1,0.0,0
2,1,1.0,0
3,1,0.0,0
4,1,0.0,0
5,1,0.0,0
6,1,1.0,0
7,2,0.0,3
8,2,4.0,3
9,2,4.0,3


In [165]:
balls['last_over_runs'].unique()

array([ 0,  3, 18,  6, 23, 10,  1,  7,  5,  4, 15, 12, 24, 14, 21,  8,  2,
       11,  9, 13, 19, 20, 17, 16, 26, 30, 22, 27, 25, 28, 33, 37, 31, 29,
       32])

In [166]:
len(balls['batsman_runs'][balls['batsman_runs']==5])

76

In [167]:
balls['batsman_runs'].unique()

array([1., 0., 4., 6., 2., 5., 3.])

In [173]:
balls.groupby(['matchId','inning','over','ball','current_score']).size().value_counts()

1    277759
2       615
Name: count, dtype: int64

In [169]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'total_runs', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'player_dismissed',
       'date', 'is_legal', 'legal_ball', 'balls_remaining', 'over_number',
       'phase', 'current_score', 'wickets_fallen', 'target', 'last_over_runs'],
      dtype='object')

In [170]:
balls.head(2)

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs
0,335982,0,0,1.0,1.0,0,0,0.0,1.0,0.0,...,2008-04-18,1,True,119,1,0,1.0,0,0.0,0
1,335982,0,0,2.0,0.0,0,0,0.0,0.0,0.0,...,2008-04-18,1,True,118,1,0,1.0,0,0.0,0


In [116]:
balls.head()

,matchId,inning,over,ball,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,...,date,is_legal,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs
0,335982,0,0,1.0,1.0,0,0,0.0,1.0,0.0,...,2008-04-18,1,True,119,1,0,1.0,0,0.0,0
1,335982,0,0,2.0,0.0,1,0,0.0,0.0,0.0,...,2008-04-18,0,False,118,1,0,2.0,0,0.0,0
2,335982,0,0,3.0,0.0,0,0,0.0,0.0,0.0,...,2008-04-18,1,True,117,1,0,2.0,0,0.0,0
3,335982,0,0,4.0,0.0,0,0,0.0,0.0,0.0,...,2008-04-18,1,True,116,1,0,2.0,0,0.0,0
4,335982,0,0,5.0,0.0,0,0,0.0,0.0,0.0,...,2008-04-18,1,True,115,1,0,2.0,0,0.0,0


In [ ]:
balls['is_boundary'] = balls['batsman_runs'].isin([4, 6]).astype(int)

def compute_balls_since_boundary(x):
    groups = x.cumsum()
    result = x.groupby(groups).cumcount()
    
    # Optional: mark pre-first-boundary as NaN
    result[groups == 0] = -1  # or np.nan
    
    return result

balls['balls_since_boundary'] = (
    balls.groupby(['matchId', 'inning'])['is_boundary']
    .transform(compute_balls_since_boundary)
)

In [56]:
balls['balls_since_boundary'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100])

In [57]:
len(balls[(balls['balls_since_boundary']>1) & (balls['over'] == 0) & (balls['ball'] == 1)])

121

In [132]:
# should only be {-1, 0}
balls[(balls['over'] == 0) & (balls['ball'] == 1)]['balls_since_boundary'].unique()

array([-1,  0,  1,  2,  3,  4,  5,  6,  7])

In [336]:
balls[balls['is_boundary'] == 1]['balls_since_boundary'].unique()

array([0])

In [337]:
balls[(balls['over'] == 0) & (balls['ball'] == 1)][
    ['is_boundary', 'balls_since_boundary']
]

,is_boundary,balls_since_boundary
0,0,-1
129,0,-1
130,0,-1
231,0,-1
355,1,0
...,...,...
278520,0,-1
278521,0,-1
278649,1,0
278772,0,-1


In [286]:
balls[(balls['over']==0) & (balls['ball']==1)].head(20)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,legal_ball,balls_remaining,over_number,phase,current_score,wickets_fallen,target,last_over_runs,is_boundary,balls_since_boundary
0,335982,0,0,1.000000,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,1.000000,...,True,119,1,0,1.000000,0,0.000000,0,0,0
129,335982,1,0,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,R Dravid,W Jaffer,AB Dinda,1.000000,...,True,119,1,0,1.000000,0,223.000000,0,0,0
130,335982,1,0,1.000000,Royal Challengers Bangalore,Kolkata Knight Riders,W Jaffer,R Dravid,AB Dinda,0.000000,...,False,119,1,0,2.000000,0,223.000000,0,0,1
231,335983,0,0,1.000000,Chennai Super Kings,Kings XI Punjab,PA Patel,ML Hayden,B Lee,0.000000,...,True,119,1,0,0.000000,0,0.000000,0,0,0
355,335983,1,0,1.000000,Kings XI Punjab,Chennai Super Kings,K Goel,JR Hopes,JDP Oram,4.000000,...,True,119,1,0,4.000000,0,241.000000,0,1,0
480,335984,0,0,1.000000,Rajasthan Royals,Delhi Daredevils,T Kohli,YK Pathan,GD McGrath,0.000000,...,True,119,1,0,0.000000,0,0.000000,0,0,0
603,335984,1,0,1.000000,Delhi Daredevils,Rajasthan Royals,G Gambhir,V Sehwag,MM Patel,0.000000,...,True,119,1,0,0.000000,0,130.000000,0,0,0
704,335985,0,0,1.000000,Mumbai Indians,Royal Challengers Bangalore,L Ronchi,ST Jayasuriya,P Kumar,0.000000,...,True,119,1,0,0.000000,0,0.000000,0,0,0
827,335985,1,0,1.000000,Royal Challengers Bangalore,Mumbai Indians,S Chanderpaul,R Dravid,A Nehra,0.000000,...,True,119,1,0,0.000000,0,166.000000,0,0,0
950,335986,0,0,1.000000,Deccan Chargers,Kolkata Knight Riders,AC Gilchrist,Y Venugopal Rao,AB Dinda,1.000000,...,True,119,1,0,1.000000,0,0.000000,0,0,0


In [204]:
balls['percentage_target_achieved'] = np.where(
    balls['inning'] == 0,
    0.0,
    balls['current_score'] / balls['target']
)
balls['percentage_target_achieved'] = (
    balls['percentage_target_achieved']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


In [205]:
balls['percentage_target_achieved'][balls['inning']==1].head(2)

129   0.004484
130   0.008969
Name: percentage_target_achieved, dtype: float64

In [206]:
matches.columns

Index(['season', 'venue', 'winner_runs', 'date', 'winner', 'team1', 'team2',
       'winner_wickets', 'player_of_match', 'matchId'],
      dtype='object')

In [207]:
balls = balls.merge(
    matches[['matchId', 'venue']],
    on='matchId',
    how='left'
)

In [208]:
balls['venue'].head(240)

0                           M Chinnaswamy Stadium
1                           M Chinnaswamy Stadium
2                           M Chinnaswamy Stadium
3                           M Chinnaswamy Stadium
4                           M Chinnaswamy Stadium
                          ...                    
235    Punjab Cricket Association Stadium, Mohali
236    Punjab Cricket Association Stadium, Mohali
237    Punjab Cricket Association Stadium, Mohali
238    Punjab Cricket Association Stadium, Mohali
239    Punjab Cricket Association Stadium, Mohali
Name: venue, Length: 240, dtype: object

In [209]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'is_legal', 'legal_ball', 'balls_remaining',
       'over_number', 'phase', 'current_score', 'wickets_fallen', 'target',
       'last_over_runs', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue'],
      dtype='object')

In [215]:
balls['balls_bowled'] = (
    balls.groupby(['matchId','inning'])['legal_ball']
    .cumsum()
)

balls['overs_bowled'] = balls['balls_bowled'] / 6

balls['current_run_rate'] = np.where(
    balls['balls_bowled'] > 0,
    balls['current_score'] / balls['overs_bowled'],
    0
)

In [ ]:
balls['runs_required'] = balls['target'] - balls['current_score']

balls['overs_remaining'] = balls['balls_remaining'] / 6

balls['required_run_rate'] = (
    balls['runs_required'] / balls['overs_remaining']
)

# handle divide by 0
balls['required_run_rate'] = balls['required_run_rate'].replace([np.inf, -np.inf], 0)
balls['required_run_rate'] = balls['required_run_rate'].fillna(0)

# cap extreme values
balls['required_run_rate'] = balls['required_run_rate'].clip(upper=36)

balls.loc[balls['inning'] == 0, 'required_run_rate'] = 0
balls.loc[balls['runs_required'] <= 0, 'required_run_rate'] = 0

In [217]:
balls[['current_run_rate','required_run_rate']].describe()

,current_run_rate,required_run_rate
count,279021.000000,279021.000000
mean,7.663732,4.849515
std,2.441020,6.321520
min,0.000000,0.000000
25%,6.391304,0.000000
50%,7.659574,0.000000
75%,8.930233,8.936170
max,66.000000,36.000000


In [222]:
balls[balls['current_run_rate']>35].head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,is_boundary,balls_since_boundary,percentage_target_achieved,venue,balls_bowled,overs_bowled,current_run_rate,runs_required,overs_remaining,required_run_rate
10465,336027,0,0,1.000000,Kolkata Knight Riders,Rajasthan Royals,Mohammad Hafeez,Salman Butt,Sohail Tanvir,4.000000,...,1,0,0.000000,Eden Gardens,1,0.166667,36.000000,-6.000000,19.833333,0.000000
10824,336028,1,0,1.000000,Mumbai Indians,Kings XI Punjab,ST Jayasuriya,SR Tendulkar,S Sreesanth,0.000000,...,0,8,0.036842,Wankhede Stadium,1,0.166667,42.000000,183.000000,19.833333,9.226891
25604,392233,0,0,1.000000,Rajasthan Royals,Kolkata Knight Riders,NV Ojha,RJ Quiney,BJ Hodge,6.000000,...,1,0,0.000000,Kingsmead,1,0.166667,36.000000,-6.000000,19.833333,0.000000
38971,419153,1,0,1.000000,Chennai Super Kings,Kolkata Knight Riders,M Vijay,ML Hayden,CH Gayle,1.000000,...,0,6,0.042857,"MA Chidambaram Stadium, Chepauk",1,0.166667,36.000000,134.000000,19.833333,6.756303
43470,501204,1,0,1.000000,Rajasthan Royals,Delhi Daredevils,R Dravid,AG Paunikar,AB Dinda,4.000000,...,1,0,0.046053,Sawai Mansingh Stadium,1,0.166667,42.000000,145.000000,19.833333,7.310924


In [149]:
balls['over'] = balls['over']/20
np.set_printoptions(suppress=True)
balls['over'].unique()

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [150]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'is_legal', 'legal_ball', 'balls_remaining',
       'over_number', 'phase', 'current_score', 'wickets_fallen', 'target',
       'last_over_runs', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue'],
      dtype='object')

In [151]:
season_map = matches.set_index('matchId')['season']
balls['season'] = balls['matchId'].map(season_map)

In [152]:
# create cyclic features
balls['sin_ball'] = np.sin(2 * np.pi * balls['ball'] / 6)
balls['cos_ball'] = np.cos(2 * np.pi * balls['ball'] / 6)

In [153]:
pd.set_option('display.float_format', '{:.6f}'.format)
balls[['ball','sin_ball','cos_ball','isWide','batsman_runs']].head(10)

,ball,sin_ball,cos_ball,isWide,batsman_runs
0,1.000000,0.866025,0.500000,0,1.000000
1,2.000000,0.866025,-0.500000,0,0.000000
2,2.000000,0.866025,-0.500000,1,0.000000
3,3.000000,0.000000,-1.000000,0,0.000000
4,4.000000,-0.866025,-0.500000,0,0.000000
5,5.000000,-0.866025,0.500000,0,0.000000
6,6.000000,-0.000000,1.000000,0,1.000000
7,1.000000,0.866025,0.500000,0,0.000000
8,2.000000,0.866025,-0.500000,0,4.000000
9,3.000000,0.000000,-1.000000,0,4.000000


In [154]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'is_legal', 'legal_ball', 'balls_remaining',
       'over_number', 'phase', 'current_score', 'wickets_fallen', 'target',
       'last_over_runs', 'is_boundary', 'balls_since_boundary',
       'percentage_target_achieved', 'venue', 'season', 'sin_ball',
       'cos_ball'],
      dtype='object')

In [155]:
balls.drop(columns=['ball','batting_team', 'bowling_team','Byes', 'LegByes', 'Penalty','over_number','is_legal', 'legal_ball','is_boundary','legal_ball','date'], inplace=True)
balls.head()

,matchId,inning,over,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,player_dismissed,...,current_score,wickets_fallen,target,last_over_runs,balls_since_boundary,percentage_target_achieved,venue,season,sin_ball,cos_ball
0,335982,0,0.000000,SC Ganguly,BB McCullum,P Kumar,1.000000,0,0,Not Out,...,1.000000,0,0.000000,0,1,0.000000,M Chinnaswamy Stadium,2008,0.866025,0.500000
1,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,1.000000,0,0.000000,0,2,0.000000,M Chinnaswamy Stadium,2008,0.866025,-0.500000
2,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,1,0,Not Out,...,2.000000,0,0.000000,0,3,0.000000,M Chinnaswamy Stadium,2008,0.866025,-0.500000
3,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,2.000000,0,0.000000,0,4,0.000000,M Chinnaswamy Stadium,2008,0.000000,-1.000000
4,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,2.000000,0,0.000000,0,5,0.000000,M Chinnaswamy Stadium,2008,-0.866025,-0.500000


In [156]:
balls.columns

Index(['matchId', 'inning', 'over', 'batsman', 'non_striker', 'bowler',
       'batsman_runs', 'isWide', 'isNoBall', 'player_dismissed', 'total_runs',
       'balls_remaining', 'phase', 'current_score', 'wickets_fallen', 'target',
       'last_over_runs', 'balls_since_boundary', 'percentage_target_achieved',
       'venue', 'season', 'sin_ball', 'cos_ball'],
      dtype='object')

In [157]:
balls['batsman_runs'] = balls['batsman_runs']/6
balls['total_runs'] = balls['total_runs']/6
balls['balls_remaining'] = balls['balls_remaining']/120
balls['wickets_fallen'] = balls['wickets_fallen']/10
balls['balls_since_boundary'] = balls['balls_since_boundary']/120

In [158]:
balls.head()

,matchId,inning,over,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,player_dismissed,...,current_score,wickets_fallen,target,last_over_runs,balls_since_boundary,percentage_target_achieved,venue,season,sin_ball,cos_ball
0,335982,0,0.000000,SC Ganguly,BB McCullum,P Kumar,0.166667,0,0,Not Out,...,1.000000,0.000000,0.000000,0,0.008333,0.000000,M Chinnaswamy Stadium,2008,0.866025,0.500000
1,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,1.000000,0.000000,0.000000,0,0.016667,0.000000,M Chinnaswamy Stadium,2008,0.866025,-0.500000
2,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,1,0,Not Out,...,2.000000,0.000000,0.000000,0,0.025000,0.000000,M Chinnaswamy Stadium,2008,0.866025,-0.500000
3,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,2.000000,0.000000,0.000000,0,0.033333,0.000000,M Chinnaswamy Stadium,2008,0.000000,-1.000000
4,335982,0,0.000000,BB McCullum,SC Ganguly,P Kumar,0.000000,0,0,Not Out,...,2.000000,0.000000,0.000000,0,0.041667,0.000000,M Chinnaswamy Stadium,2008,-0.866025,-0.500000


In [159]:
balls['current_score'] = balls['current_score']/200
balls['target'] = balls['target']/200
balls['last_over_runs'] = balls['last_over_runs']/200

In [160]:
balls.to_csv('data3.csv')